## **RAG-Based Semester Course Advisor**

Recommends which courses a student should register for next semester, using retrieved evidence from the course catalog and the registration rules, explicit Python rule checks, and an LLM chatbot.
All course data and registration rules are **fictional sample data** created for this assignment.

**Pipeline:** student profile -> search query -> retrieve course + rule passages -> rule checker accepts/rejects -> LLM explains the result with citations -> BM chatbot.


### 0. Setup


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():  # allow running from a sub-folder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "tests"))

import kb_loader, rules
from kb_loader import load_courses, load_rules, validate_kb
import importlib.util

profile_path = ROOT / "src" / "profile.py"
spec = importlib.util.spec_from_file_location("course_advisor_profile", profile_path)
profile_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = profile_module
spec.loader.exec_module(profile_module)

load_profile = profile_module.load_profile
prompt_profile = profile_module.prompt_profile
from rules import check_course, check_plan, format_plan_result

print("Project folder:", ROOT)

Project folder: c:\MESK\assignments\KB-AI


### 1. Knowledge base

checks that every course has all required fields, every prerequisite code exists, codes are unique, there are no prerequisite cycles, every course has a 4-6 sentence description, and the rules are numbered R1..R9.


In [3]:
print(validate_kb())
courses = load_courses()
rules_doc = load_rules()

{'courses': 28, 'descriptions': 28, 'rules': 9}


In [4]:
print(f"{'Code':8} {'Title':38} {'Cr':>2} {'Lv':>2}  {'Offered':12} Prerequisites")
for c in courses.values():
    print(
        f"{c.code:8} {c.title:38} {c.credits:>2} {c.level:>2}  {'/'.join(c.offered):12} {', '.join(c.prerequisites) or '-'}"
    )

Code     Title                                  Cr Lv  Offered      Prerequisites
CS101    Introduction to Programming             3  1  Fall         -
MATH101  Calculus I                              3  1  Fall         -
PHYS101  Physics for Computing                   3  1  Fall/Spring  -
ENG101   Technical Writing                       2  1  Fall/Spring  -
MATH102  Discrete Mathematics                    3  1  Spring       -
CS102    Object-Oriented Programming             3  1  Spring       CS101
CS201    Data Structures and Algorithms          3  2  Fall         CS102
MATH201  Linear Algebra                          3  2  Fall         MATH101
MATH202  Probability and Statistics              3  2  Spring       MATH101
CS202    Databases                               3  2  Spring       CS102
CS203    Computer Organization                   4  2  Spring       CS101, PHYS101
CS204    Web Development                         3  2  Fall         CS102
CS205    Operating Systems           

Two pairs of courses clash in time (one identical slot, one partial overlap). The rule checker must catch both:


In [5]:
for sem in ("Fall", "Spring"):
    for a, b, details in rules.catalog_clashes(sem):
        print(f"{sem}: {a} and {b}")
        for d in details:
            print("   ", d)

Fall: CS301 and CS304
    CS301 Sun 12:30-14:00 overlaps CS304 Sun 12:30-14:00
    CS301 Tue 12:30-14:00 overlaps CS304 Tue 12:30-14:00
Spring: CS305 and CS306
    CS305 Tue 12:30-14:00 overlaps CS306 Tue 13:30-15:00


The registration rules (fictional policy) that the checker enforces:


In [6]:
for rid, r in rules_doc.items():
    print(f"{rid} - {r['title']}\n    {r['text']}\n")

R1 - Credit-hour range
    In a regular semester a student must register for at least 12 and at most 18 credit hours. A semester load below 12 credit hours is not allowed unless rule R3 applies. The limits of R2 and R3 replace the maximum of 18 for the students they describe.

R2 - Overload for high GPA
    A student with a cumulative GPA of 3.50 or higher may register for up to 21 credit hours. The minimum of 12 credit hours from R1 still applies. A GPA below 3.50 gives no overload, and the maximum stays at 18 credit hours.

R3 - Academic probation
    A student with a cumulative GPA below 2.00 is on academic probation. A student on probation must register for at least 9 and at most 12 credit hours. R3 replaces both R1 and R2 for this student.

R4 - Level eligibility
    A student may register only for courses whose level is equal to or lower than the student's own level. A student in level 3 may take level 1, 2, and 3 courses, but not level 4 courses. There are no exceptions to this 

### 2. Student profiles

Profiles are validated when loaded. A missing or invalid value is **flagged (rule R9)**, never guessed. The six synthetic profiles cover a normal strong student, an unmet prerequisite, probation (low GPA), a credit-limit case, a time clash and missing data.


In [7]:
import json

expected = json.loads((ROOT / "profiles" / "expected_results.json").read_text())
profiles = {}
for e in expected:
    p = load_profile(ROOT / "profiles" / e["file"], courses)
    profiles[e["profile"]] = p
    print(
        f"{p.id}  {p.name:16} level={p.level}  gpa={p.gpa}  semester={p.semester}  desired={p.desired_credits}  passed={len(p.passed or [])} courses"
    )
    print(f"     scenario: {e['scenario']}")
    for f in p.flags:
        print("     FLAG:", f)

S01  Nour Hassan      level=3  gpa=3.82  semester=Fall  desired=13  passed=12 courses
     scenario: Normal strong student (level 3, GPA 3.82, Fall)
S02  Omar Khaled      level=3  gpa=2.9  semester=Fall  desired=15  passed=11 courses
     scenario: UNMET PREREQUISITE (level 3, GPA 2.9, Fall) - MATH202 never passed
S03  Salma Adel       level=3  gpa=1.8  semester=Fall  desired=15  passed=12 courses
     scenario: Low-GPA probation student (level 3, GPA 1.8, Fall)
S04  Youssef Mostafa  level=4  gpa=3.2  semester=Fall  desired=21  passed=18 courses
     scenario: CREDIT LIMIT EXCEEDED (level 4, GPA 3.2, Fall, wants 21 credits)
S05  Laila Ibrahim    level=4  gpa=3.6  semester=Spring  desired=18  passed=17 courses
     scenario: TIME CLASH (level 4, GPA 3.6, Spring)
S06  Karim Samir      level=None  gpa=3.0  semester=Fall  desired=None  passed=4 courses
     scenario: MISSING DATA (level missing, desired_credits missing, unknown passed code CS999)
     FLAG: R9: missing field 'level' - leve

Interactive input for a new student (here with scripted answers so the notebook runs top to bottom; replace `input_fn` with `input` to type answers):


In [8]:
answers = iter(
    [
        "S99",
        "Mona Ali",
        "2",
        "3.4",
        "CS101, CS102, MATH101, PHYS101, ENG101, MATH102",
        "python",
        "AI, data",
        "15",
        "Fall",
    ]
)
new_student = prompt_profile(
    input_fn=lambda q: (print(q, end=""), a := next(answers), print(a))[1],
    courses=courses,
)
print(new_student)

Student profile (press Enter to skip any question)
Student ID: S99
Name: Mona Ali
Level (1-4): 2
Cumulative GPA (0.0-4.0): 3.4
Passed course codes, comma-separated (type 'none' if you have passed none): CS101, CS102, MATH101, PHYS101, ENG101, MATH102
Skills, comma-separated: python
Interests / goals, comma-separated: AI, data
Desired credit hours: 15
Semester to register for (Fall/Spring): Fall
Profile(id='S99', name='Mona Ali', level=2, gpa=3.4, passed=['CS101', 'CS102', 'MATH101', 'PHYS101', 'ENG101', 'MATH102'], skills=['python'], interests=['AI', 'data'], desired_credits=15, semester='Fall', flags=[])


### 3. Rule checker on the six profiles

Each profile is checked with `check_plan(profile, candidate_plan)`. Courses are processed in the order listed, so an earlier course has priority. Every rejection shows the reason and the rule ID.


In [9]:
for e in expected:
    p = profiles[e["profile"]]
    print("=" * 78)
    print(f"{e['profile']} - {e['scenario']}")
    print("Candidate plan:", ", ".join(e["candidate_plan"]))
    print(format_plan_result(check_plan(p, e["candidate_plan"]), courses))
    print()

S01 - Normal strong student (level 3, GPA 3.82, Fall)
Candidate plan: CS301, CS302, CS205, CS307
Credit limits: 12-21 (R2, status: overload)
ACCEPTED (13 credit hours):
  + CS301  Machine Learning (3 cr)
  + CS302  Software Engineering (3 cr)
  + CS205  Operating Systems (3 cr)
  + CS307  Embedded Systems (4 cr)
REJECTED:
  (none)
FLAGS:
  (none)

S02 - UNMET PREREQUISITE (level 3, GPA 2.9, Fall) - MATH202 never passed
Candidate plan: CS301, CS304, CS302, CS205, CS307, CS402
Credit limits: 12-18 (R1, status: standard)
ACCEPTED (13 credit hours):
  + CS304  Image Processing and Computer Vision (3 cr)
  + CS302  Software Engineering (3 cr)
  + CS205  Operating Systems (3 cr)
  + CS307  Embedded Systems (4 cr)
REJECTED:
  - CS301  Machine Learning  [R5]
        R5: prerequisite(s) not passed: MATH202 (required for CS301)
  - CS402  Cybersecurity  [R4, R5]
        R4: CS402 is a level 4 course but the student is in level 3
        R5: prerequisite(s) not passed: CS303 (required for CS402)


### 4. Rule-checker tests

One pass case and one fail case for every rule (R1-R9), KB validation tests, profile validation tests, and a comparison of all six profiles with the hand-written expected results (`profiles/expected_results.md`). The same tests run with `pytest tests/`.


In [ ]:
# Use the project profile module when test_rules imports "profile".
sys.modules["profile"] = profile_module

test_rules_path = ROOT / "tests" / "test_rules.py"
test_rules_spec = importlib.util.spec_from_file_location("test_rules", test_rules_path)
test_rules = importlib.util.module_from_spec(test_rules_spec)
sys.modules["test_rules"] = test_rules
test_rules_spec.loader.exec_module(test_rules)

passed, failed = test_rules.run_all(verbose=False)
for name in passed:
    print("PASS", name)
for name, tb in failed:
    print("FAIL", name)
    print(tb)
print(f"\n{len(passed)} passed, {len(failed)} failed")
assert not failed

PASS test_R1_credit_range_pass_and_fail
PASS test_R1_maximum_rejects_course_with_rule_id
PASS test_R2_overload_pass_and_fail
PASS test_R3_probation_pass_and_fail
PASS test_R4_level_pass_and_fail
PASS test_R5_prerequisites_pass_and_fail
PASS test_R6_offered_semester_pass_and_fail
PASS test_R7_time_clash_pass_and_fail
PASS test_R8_no_retake_pass_and_fail
PASS test_R9_missing_data_is_flagged_never_guessed
PASS test_all_six_profiles_match_expected_results
PASS test_check_plan_contract_shape
PASS test_credit_numbers_in_code_appear_in_rules_document
PASS test_duplicates_and_lowercase_codes_are_handled
PASS test_eligible_courses_matches_check_course
PASS test_every_rejection_has_a_documented_rule_id
PASS test_every_rule_id_used_by_checker_is_documented
PASS test_interactive_prompt_complete_and_partial
PASS test_kb_descriptions_have_4_to_6_sentences
PASS test_kb_has_chained_prerequisites_and_clashes
PASS test_kb_is_valid_and_meets_size_requirements
PASS test_load_courses_raises_on_invalid_cata

### 5. RAG pipeline

### 5.1 Parsing and chunking

Each course becomes one chunk (catalog facts + description) and each rule R1-R9 becomes one chunk, plus one chunk for the _Terms_ section. Every chunk keeps its source (e.g. `registration_rules.md#R5`) so recommendations can cite it. Longer documents would be split into overlapping 200-word windows by `chunk_text()`.


In [11]:
import os
from collections import Counter
from chunker import build_chunks, chunk_text

chunks = build_chunks()
print(len(chunks), "chunks:", dict(Counter(c.kind for c in chunks)))
for c in (chunks[13], next(c for c in chunks if c.ref == "R5")):
    print("-" * 78)
    print(c.id, "|", c.source)
    print(c.text)

38 chunks: {'course': 28, 'rule': 9, 'terms': 1}
------------------------------------------------------------------------------
course:CS301#0 | course_descriptions.md#CS301
CS301 - Machine Learning. Level 3 course, 3 credit hours. Offered in: Fall. Prerequisites: CS201, MATH201, MATH202. Meets: Sun 12:30-14:00; Tue 12:30-14:00. Topics: AI, data.
Machine Learning introduces supervised and unsupervised learning, including linear models, decision trees, support vector machines, and clustering. Students study model evaluation, overfitting, regularisation, and feature engineering. Practical work uses Python and scikit-learn on real datasets. The course ends with a project that compares several models on a problem chosen by the student. It is the entry point to deep learning and applied AI.
------------------------------------------------------------------------------
rule:R5#0 | registration_rules.md#R5
Registration rule R5 - Prerequisite policy. Every prerequisite listed for a course must

### 5.2 Embeddings and vector store

Embeddings: `sentence-transformers/all-MiniLM-L6-v2` (downloaded once). If it cannot be loaded (no internet), the code falls back to TF-IDF embeddings automatically; set `ADVISOR_EMBEDDER=tfidf` to force the fallback.
Vector store: FAISS `IndexFlatIP` on normalised vectors (cosine similarity), saved in `index/` and rebuilt automatically when a knowledge-base file changes.


In [ ]:
from vector_store import get_store

store = get_store(rebuild=True)
print(
    "Embedder:",
    store.embedder.name,
    "| backend:",
    store.backend,
    "| vectors:",
    store.vectors.shape,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[vector_store] Built index: 38 chunks, embedder=minilm:sentence-transformers/all-MiniLM-L6-v2, backend=faiss, saved to C:\MESK\assignments\KB-AI\index
Embedder: minilm:sentence-transformers/all-MiniLM-L6-v2 | backend: faiss | vectors: (38, 384)


### 5.3 Retrieval

`retrieve(query, k)` returns the top-k passages with their source and cosine score. For a student, `profile_query()` builds the query from interests, skills, level and semester, and `rank_courses()` ranks every course the student has not passed yet.


In [ ]:
from retriever import retrieve, profile_query, rank_courses

for q in [
    "computer vision and image analysis",
    "What happens if my GPA is very low?",
    "Can I take two courses at the same time slot?",
]:
    print(f"Query: {q}")
    for h in retrieve(q, k=3):
        print(f"   {h['score']:.3f}  {h['source']}")
    print()

p = profiles["S05"]
print("Profile query for S05:", profile_query(p))
print("Top-ranked courses:", rank_courses(p)[:6])

Query: computer vision and image analysis
   0.536  course_descriptions.md#CS304
   0.313  course_descriptions.md#CS301
   0.290  course_descriptions.md#MATH201

Query: What happens if my GPA is very low?
   0.484  registration_rules.md#R2
   0.437  registration_rules.md#R3
   0.346  registration_rules.md#terms

Query: Can I take two courses at the same time slot?
   0.511  registration_rules.md#R7
   0.462  registration_rules.md#R5
   0.431  registration_rules.md#R6

Profile query for S05: NLP. natural language processing NLP text. deep learning. mobile apps. mobile application development android. python. pytorch. android. level 4 course. offered in Spring
Top-ranked courses: [('CS305', 0.6036), ('CS403', 0.5384), ('CS401', 0.4831), ('CS306', 0.3702), ('CS310', 0.3498), ('CS404', 0.3047)]


### 6. LLM setup

The LLM only **explains** the rule checker's decisions; it never decides eligibility. Set one API key before running this cell (a free Groq or Gemini key is enough). With no key, the notebook still runs and prints template answers built from the same evidence.


In [14]:
# os.environ["GROQ_API_KEY"] = "paste-your-key-here"
from generator import get_llm, SYSTEM_PROMPT

llm = get_llm()
print("LLM in use:", llm.name)
print()
print("System prompt:")
print(SYSTEM_PROMPT)

LLM in use: offline (template answers, no LLM key set)

System prompt:
You are a university course-registration advisor.
Rules you must follow:
1. Use ONLY the evidence passages and the rule-checker results given to you. Do not invent courses, rules, grades or numbers.
2. The rule checker's ACCEPTED / REJECTED decisions are final. Never recommend a rejected course and never question an accepted one.
3. Cite the source of every fact in square brackets exactly as written, for example [course_descriptions.md#CS301] or [registration_rules.md#R5].
4. If information is missing or flagged, say so clearly and tell the student what to provide. Never guess it.
5. Be concise, friendly and specific to this student's interests, skills and goals.


### 7. Recommendations for the six test profiles

Each student asks for the courses in their candidate plan (from `profiles/expected_results.json`). For every profile the output shows the **input**, the **retrieved evidence**, the **accepted** courses with reasons, eligibility and sources, the **excluded** courses with reasons, the **total credit hours**, the flags, and the advisor's message.

Required cases: **S02 = unmet prerequisite** (CS301 needs MATH202), **S04 = credit limit exceeded** (wants 21, allowed 18). Also: S03 probation, S05 time clash, S06 missing data.
All outputs are also saved to `outputs/`.


In [ ]:
import json
from pathlib import Path
from recommender import recommend, format_recommendation

OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
results = {}

for e in expected:
    p = profiles[e["profile"]]
    rec = recommend(p, requested=e["candidate_plan"], llm=llm)
    results[e["profile"]] = rec
    header = f"{e['profile']} - {e['scenario']}"
    text = (
        "=" * 90
        + "\n"
        + header
        + "\n"
        + "=" * 90
        + "\nINPUT PROFILE\n"
        + json.dumps(p.to_dict(), indent=1)
        + "\n\n"
        + format_recommendation(rec)
    )
    (OUT / f"{e['profile']}_output.txt").write_text(text, encoding="utf-8")
    print(text)
    # agreement with Member A's hand-written expected results
    got = [c["code"] for c in rec["courses"]]
    assert got == e["accepted"] and rec["total_credits"] == e["total_credits"], (
        e["profile"],
        got,
    )
    print(f"\n[check] accepted list matches expected_results.json: {got}\n\n")

(OUT / "all_results.json").write_text(json.dumps(results, indent=1), encoding="utf-8")
print("Saved", len(results), "results to", OUT)

S01 - Normal strong student (level 3, GPA 3.82, Fall)
INPUT PROFILE
{
 "id": "S01",
 "name": "Nour Hassan",
 "level": 3,
 "gpa": 3.82,
 "passed": [
  "CS101",
  "CS102",
  "MATH101",
  "PHYS101",
  "ENG101",
  "MATH102",
  "CS201",
  "MATH201",
  "MATH202",
  "CS202",
  "CS203",
  "CS204"
 ],
 "skills": [
  "python",
  "linear algebra"
 ],
 "interests": [
  "AI",
  "computer vision"
 ],
 "desired_credits": 13,
 "semester": "Fall"
}

STUDENT S01 - Nour Hassan   (mode: requested, LLM: offline template)
Requested courses: CS301, CS302, CS205, CS307
Search query: AI. artificial intelligence AI machine learning. computer vision. computer vision image processing. python. linear algebra. level 3 course. offered in Fall

RETRIEVED EVIDENCE (top passages for this student):
  [0.667] course_descriptions.md#MATH201: MATH201 - Linear Algebra. Level 2 course, 3 credit hours. Offered in: Fall. Prerequisites: MATH101. Meets: Mon 11:00-12:30; Wed 11:00-12:30. Topics: math, AI. Linear Algebra covers ve

### Summary of the six demonstrations


In [ ]:
print(f"{'ID':4} {'Scenario':55} {'Accepted':32} {'Cr':>3}  Rejected")
for e in expected:
    r = results[e["profile"]]
    acc = ", ".join(c["code"] for c in r["courses"]) or "-"
    rej = (
        ", ".join(f"{x['code']}({'/'.join(x['rule_refs'])})" for x in r["excluded"])
        or "-"
    )
    print(
        f"{e['profile']:4} {e['scenario'][:55]:55} {acc:32} {r['total_credits']:>3}  {rej}"
    )

ID   Scenario                                                Accepted                          Cr  Rejected
S01  Normal strong student (level 3, GPA 3.82, Fall)         CS301, CS302, CS205, CS307        13  -
S02  UNMET PREREQUISITE (level 3, GPA 2.9, Fall) - MATH202 n CS304, CS302, CS205, CS307        13  CS301(R5), CS402(R4/R5)
S03  Low-GPA probation student (level 3, GPA 1.8, Fall)      CS301, CS302, CS205                9  CS307(R3), CS304(R7)
S04  CREDIT LIMIT EXCEEDED (level 4, GPA 3.2, Fall, wants 21 CS402, CS404, CS490, CS307, CS304, HUM401  18  CS204(R1)
S05  TIME CLASH (level 4, GPA 3.6, Spring)                   CS305, CS303, CS310, CS401, CS403, HUM401  17  CS306(R7)
S06  MISSING DATA (level missing, desired_credits missing, u -                                  0  CS101(R8/R9), CS201(R9), CS204(R9), XYZ123(R9)


### 8. The advisor's own plan (auto mode)

Here the student does not name any course. The advisor tries courses in order of relevance to the student's interests (retrieval score) and keeps a course only if the rule checker accepts it, until the desired credit load is reached. The most relevant courses that were left out are listed with the reason.


In [ ]:
for pid, p in profiles.items():
    rec = recommend(p, llm=llm)
    lim = rec["credit_limits"]
    print(
        f"{pid} {p.name} | interests: {', '.join(p.interests)} | desired {p.desired_credits} cr, allowed {lim['min']}-{lim['max']}"
    )
    for c in rec["courses"]:
        print(f"   + {c['code']:7} {c['title']:42} {c['credits']} cr  - {c['why']}")
    print(f"   total: {rec['total_credits']} credit hours")
    for x in rec["excluded"][:4]:
        print(f"   - {x['code']:7} {x['reasons'][0][:95]}")
    for f in rec["flags"]:
        print(f"   ! {f[:110]}")
    print()

S01 Nour Hassan | interests: AI, computer vision | desired 13 cr, allowed 12-21
   + CS301   Machine Learning                           3 cr  - matches your AI, python (topics: AI, data); relevance 0.69
   + CS307   Embedded Systems                           4 cr  - broadens your study plan (topics: hardware, embedded, IoT); relevance 0.44
   + CS302   Software Engineering                       3 cr  - broadens your study plan (topics: software, teamwork); relevance 0.36
   + CS205   Operating Systems                          3 cr  - broadens your study plan (topics: systems); relevance 0.31
   total: 13 credit hours
   - CS304   R7: time clash with CS301 (CS304 Sun 12:30-14:00 overlaps CS301 Sun 12:30-14:00; CS304 Tue 12:3
   - CS306   R6: CS306 is not offered in Fall (offered: Spring)
   - CS401   R4: CS401 is a level 4 course but the student is in level 3
   - CS310   R6: CS310 is not offered in Fall (offered: Spring)

S02 Omar Khaled | interests: AI, machine learning | desired 15 c

### 9. Chatbot conversation - BM

**BM** is the team's registration chatbot. A **new student** is interviewed in the chat (name, level, semester, passed courses, GPA, skills, interests/goals, desired credit load), so any student can enter their own profile; the six demo students can also be opened directly. Course questions are answered by the rule checker (always correct, with the rule cited) and open questions by the LLM over the retrieved passages. Every factual reply lists its sources.

### 9.1 A new student enters their own profile in the chat


In [18]:
from chatbot import ChatSession

new = ChatSession.new_student(llm)
print("BM:", new.messages[0]["content"])
print("-" * 90)
for msg in [
    "Hi",
    "My name is Ahmed Ali",
    "third year",
    "Fall",
    "CS101, CS102, MATH101, MATH102, ENG101, PHYS101, CS201, MATH201, CS202, CS203",
    "3.4",
    "python, problem solving",
    "AI and data science",
    "15",
    "can I take CS301?",
    "what courses are offered in level 4?",
    "thanks!",
]:
    print("STUDENT:", msg)
    print("BM:", new.ask(msg))
    print("-" * 90)

BM: Hi, I'm **BM**, your course-registration assistant. I'll ask you a few quick questions about your studies, then suggest courses that fit your goals and follow every registration rule.

First, **what's your name?**
------------------------------------------------------------------------------------------
STUDENT: Hi
BM: Hello! Happy to help.

What's your name?
------------------------------------------------------------------------------------------
STUDENT: My name is Ahmed Ali
BM: Nice to meet you, Ahmed! **Which academic level are you in?** (1, 2, 3 or 4)
------------------------------------------------------------------------------------------
STUDENT: third year
BM: **Which semester are you registering for?** Fall or Spring?
------------------------------------------------------------------------------------------
STUDENT: Fall
BM: **Which courses have you already passed?** Type the course codes separated by commas (for example: CS101, MATH101, ENG101), or **none** if this is y

### 9.2 Demo student S02 (the unmet-prerequisite case)


In [19]:
session = ChatSession(profiles["S02"], llm)
for msg in [
    "Which courses should I take this semester?",
    "Why was CS301 rejected?",
    "Can I take CS402?",
    "What is the credit limit for me?",
    "I passed MATH202",
    "drop CS304",
    "add CS301",
    "show my plan",
]:
    print("STUDENT:", msg)
    print("BM:", session.ask(msg))
    print("-" * 90)

STUDENT: Which courses should I take this semester?
BM: Hi Omar, based on your interest in AI, machine learning and the courses you have passed, these 4 course(s) (13 credit hours) are allowed for you this Fall:
  * CS304 Image Processing and Computer Vision: matches your AI, python (topics: AI, vision); relevance 0.49 [course_descriptions.md#CS304]
  * CS307 Embedded Systems: broadens your study plan (topics: hardware, embedded, IoT); relevance 0.43 [course_descriptions.md#CS307]
  * CS302 Software Engineering: broadens your study plan (topics: software, teamwork); relevance 0.41 [course_descriptions.md#CS302]
  * CS205 Operating Systems: broadens your study plan (topics: systems); relevance 0.32 [course_descriptions.md#CS205]
  x CS301 not included: R5: prerequisite(s) not passed: MATH202 (required for CS301) [registration_rules.md#R5]
  x CS306 not included: R6: CS306 is not offered in Fall (offered: Spring); R5: prerequisite(s) not passed: MATH202 (required for CS306) [registration

### 9.3 Missing data (student S06): BM flags instead of guessing


In [20]:
s6 = ChatSession(profiles["S06"], llm)
for msg in ["recommend courses for me", "my level is 2", "recommend courses for me"]:
    print("STUDENT:", msg)
    print("BM:", s6.ask(msg)[:1500])
    print("-" * 90)

STUDENT: recommend courses for me
BM: Hi Karim, I cannot recommend any course yet.
  x CS204 not included: R9: student level is missing, so level eligibility (R4) of CS204 cannot be verified [registration_rules.md#R9]
  x CS202 not included: R9: student level is missing, so level eligibility (R4) of CS202 cannot be verified; R6: CS202 is not offered in Fall (offered: Spring) [registration_rules.md#R9] [registration_rules.md#R6]
  x CS403 not included: R9: student level is missing, so level eligibility (R4) of CS403 cannot be verified; R6: CS403 is not offered in Fall (offered: Spring); R5: prerequisite(s) not passed: CS204, CS202 (required for CS403) [registration_rules.md#R9] [registration_rules.md#R6] [registration_rules.md#R5]
  x CS307 not included: R9: student level is missing, so level eligibility (R4) of CS307 cannot be verified; R5: prerequisite(s) not passed: CS203 (required for CS307) [registration_rules.md#R9] [registration_rules.md#R5]
  x CS310 not included: R9: student le

### 9.4 Open the BM web interface


In [26]:
from chatbot import launch_app, run_console_chat, load_demo_profiles

LAUNCH_UI = True   # set to True to open the BM chat window

if LAUNCH_UI:
    launch_app(load_demo_profiles(ROOT / "profiles"), llm, share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
